# UE Data Mining - Projet

**Public_records**

In [ ]:
# TODO
!git clone https://github.com/Sheik-info/20262027-UE-DataMining-Project.git
%cd 20262027-UE-DataMining-Project

In [ ]:
!git pull

In [ ]:
import pandas as pd

In [ ]:
df_pr = pd.read_csv("data/public_records/institutional_memos.csv")  #à finir quand on saura faire les textes
df_lf = pd.read_csv("data/public_records/licensed_facilities.csv") #à finir quand on saura faire les textes
df_lp = pd.read_csv("data/public_records/licensed_professionals.csv")
df_of = pd.read_csv("data/public_records/officer_filings.csv")
df_id = pd.read_csv("data/public_records/public_and_investigative_documents.csv")
df_pa = pd.read_csv("data/public_records/public_appointments.csv")
df_re = pd.read_csv("data/public_records/registered_entities.csv")
df_rce = pd.read_csv("data/public_records/regulatory_case_events.csv")
df_sac = pd.read_csv("data/public_records/site_access_case_events.csv")

institutional_memos.csv

## Nettoyage — `institutional_memos.csv`

Ce fichier contient des notes et mémos institutionnels courts, caractérisés par un champ textuel très dense, complexe et difficile à exploiter directement sous sa forme brute.

**Problèmes identifiés et étapes de traitement :**
1. **Extraction d'informations textuelles :** Analyse de la structure du champ textuel libre pour extraire et catégoriser le type de document (par exemple : `SUBJECT`, `SCOPE`, `MEETING`, `REQUEST`, `ROUTE`, `PERIOD`, ou `OTHER`) afin de structurer l'information.
2. **Identification des colonnes constantes :** Repérage et suppression automatique des colonnes sans pouvoir prédictif ou informatif (celles possédant une seule ou aucune valeur unique).
3. **Gestion des valeurs manquantes :** Évaluation globale du taux de complétude des variables de la table pour éliminer le bruit et conserver uniquement les attributs exploitables.

In [ ]:
print(df_pr.head())

# Valeurs manquantes
print(df_pr.isna().sum())

# Nombre de valeurs uniques
print(df_pr.nunique())

In [ ]:
print(df_pr.head())

# Valeurs manquantes
print(df_pr.isna().sum())

# Nombre de valeurs uniques
print(df_pr.nunique())

In [ ]:
constant_cols = [
    col for col in df.columns
    if df_pr[col].nunique() <= 1
]

df_pr = df_pr.drop(columns=constant_cols)

In [ ]:
for col in df_pr.select_dtypes(include="object").columns:
    print("\n---", col, "---")
    print("Nombre de catégories :", df_pr[col].nunique())
    print(df_pr[col].value_counts().head(10))

In [ ]:
# def extract_document_type(text):
#     text = str(text).strip().upper()

#     if text.startswith("SUBJECT"):
#         return "SUBJECT"
#     elif text.startswith("SCOPE"):
#         return "SCOPE"
#     elif text.startswith("MEETING RECORD"):
#         return "MEETING"
#     elif text.startswith("REQUEST"):
#         return "REQUEST"
#     elif text.startswith("ROUTE"):
#         return "ROUTE"
#     elif text.startswith("PERIOD"):
#         return "PERIOD"
#     else:
#         return "OTHER"


# df_pr["document_type"] = df["text"].apply(extract_document_type)
# df_pr = df_pr.drop('text', axis=1)
df_pr.to_csv("data/public_records/institutional_memos.csv", index=False)

licensed_facilities.csv

on enleve cette tab car elle est vide elle ne contient rien

In [ ]:
df_lf = df_lf.drop('license_end', axis=1)
df_lf = df_lf.drop('facility_status', axis=1)

In [ ]:
print(df_lf.head)

licensed_professionals.csv

In [ ]:
# Valeurs manquantes
# print(df_lp.isna().sum())
# print(len(df_lp))

# Nombre de valeurs uniques
print(df_lp.nunique())

In [ ]:
print(df_lp.columns)

on a décidé de les supprimer car ils ont tous la même valeur ou sont vides

In [ ]:
df_lp = df_lp.drop('license_status', axis=1)
df_lp = df_lp.drop('license_expiry_date', axis=1)

In [ ]:
df_lp['license_issue_date'] = pd.to_datetime(df_lp['license_issue_date'], format='%Y-%m-%d', errors='coerce')
df_lp[df_lp['license_issue_date'].isna()]
df_lp.loc[26, 'license_issue_date'] = pd.to_datetime('1994-07-01', format='%Y-%m-%d')

In [ ]:
df_lp['license_issue_date'] = pd.to_datetime(df_lp['license_issue_date'], format='%Y-%m-%d', errors='coerce')

date_par_defaut = df_lp['license_issue_date'].min()
df_lp['license_issue_date'] = df_lp['license_issue_date'].fillna(date_par_defaut)

print(f"Date par défaut appliquée automatiquement aux erreurs : {date_par_defaut.strftime('%Y-%m-%d')}")
print(df_lp.info())

In [ ]:
print(df_lp.iloc[26])

officer_filings.csv


on enleve le FLG- qui ne sert pas à grand chose
REG- et OFR aussi

In [ ]:
print(df_of.columns)

In [ ]:
#pas d'erreur c'est juste qu'on peut le faire qu'une fois
df_of['filing_id'] = df_of['filing_id'].str.replace("FLG-", "", regex=False).astype(int)
print(df_of.head())

In [ ]:
df_of['registration_number'] = df_of['registration_number'].str.replace("REG-", "", regex=False)
print(df_of.head())

In [ ]:
df_of['officer_local_ref'] = df_of['officer_local_ref'].str.replace("OFR-", "", regex=False)
print(df_of.head())

mettre les dates dans le bon format

In [ ]:
df_of['appointment_date'] = pd.to_datetime(df_of['appointment_date'], format='%Y-%m-%d', errors='coerce')

date_par_defaut = df_of['appointment_date'].min()
df_of['appointment_date'] = df_of['appointment_date'].fillna(date_par_defaut)

print(f"Date par défaut appliquée automatiquement aux erreurs : {date_par_defaut.strftime('%Y-%m-%d')}")
print(df_of.info())

birth year or  age band donnne des données trop incohérentes on les divise

In [ ]:
#conversion en chaîne de caractères pour le traitement
serie_temp = df_of['birth_year_or_age_band'].astype(str).str.strip()

#année de naissance 4 chiffres
df_of['birth_year'] = np.where(serie_temp.str.match(r'^\d{4}$'), serie_temp, np.nan)

#tranche d'âge (valeur contenant un tiret, ex: 1960-1969)
df_of['age_band'] = np.where(serie_temp.str.contains('-', regex=False), serie_temp, np.nan)

df_of = df_of.drop('birth_year_or_age_band', axis=1)
print(df_of)

In [ ]:
#s'assurer de la conversion des colonnes au format Datetime
app_date = pd.to_datetime(df_of['appointment_date'], errors='coerce')
cess_date = pd.to_datetime(df_of['cessation_date'], errors='coerce')

#calculer la durée des mandats uniquement pour les lignes où la date de cessation existe
durees = (cess_date - app_date).dt.days
durees_valides = durees.dropna()

#Calculer la moyenne et la médiane
moyenne_jours = durees_valides.mean()
mediane_jours = durees_valides.median()

print(f"Nombre de mandats terminés analysés : {len(durees_valides)}")
print(f"Durée moyenne des mandats  : {moyenne_jours:.1f} jours (environ {moyenne_jours/365.25:.1f} ans)")
print(f"Durée médiane des mandats  : {mediane_jours:.1f} jours (environ {mediane_jours/365.25:.1f} ans)")

In [ ]:

appointment_dt = pd.to_datetime(df_of['appointment_date'], errors='coerce')
cessation_dt = pd.to_datetime(df_of['cessation_date'], errors='coerce')

date_projection = appointment_dt + pd.DateOffset(years=9)

df_of['cessation_date'] = cessation_dt.fillna(date_projection)

display(df_of[['appointment_date', 'cessation_date']].head(10))

public_and_investigative_documents.csv


public_appointments.csv

registered_entities.csv


## Lecture et exploration des données

In [ ]:
df_re.head()

In [ ]:
df_re.info()

In [ ]:
df_re.describe()

## Recherche des valeurs manquantes

In [ ]:
df_re.isnull().sum()
#(df.isnull().sum() / len(df)) * 100

### Nous remarquons que "parent_registration_number" est 100% vide, on le supprime

> Add blockquote



In [ ]:
df_re=df_re.drop(columns=["parent_registration_number"])

In [ ]:
df_re.head()

## Recherche et suppression des doublons

In [ ]:
df_re.duplicated().sum()
#df_re[df.duplicated(keep=False)]

# regulatory_case_events.csv


### Lecture et exploration des données

In [ ]:
df_rce.info()

In [ ]:
df_rce.describe()

In [ ]:
df_rce.nunique()

ON remaque qu'il y'a beaucoup de valeurs manquante en plus c est categoriel

## Recherche et suppression des doublons

In [ ]:
df_rce.duplicated().sum()

In [ ]:
df_rce[df_rce.duplicated(keep=False)]

In [ ]:
df_rce=df_rce.drop_duplicates()
#x=df_rce.iloc[175]
#print(x)

## Recherche des valeurs manquantes

In [ ]:
df_rce.isnull().sum()

In [ ]:
#Pourcentage de valeurs manquantes
 (df_rce.isnull().sum() / len(df_rce)) * 100

site_access_case_events.csv